# Customer Churn Prediction

In [ ]:
import pandas as pd

In [ ]:
RANDOM_STATE = 42

## 1. Data Understanding & Preparation 

In [ ]:
# load dataset
df = pd.read_csv('../data/TelcoCustomerChurn.csv')

### Data type & structure checks

In [ ]:
df.head()

In [ ]:
df.info()
df.head()

### Missing values analysis

In [ ]:
# Standard null check
df.isnull().sum()
# => No null values found in the dataset.

In [ ]:
# find all string columns and check for empty strings
string_columns = df.select_dtypes(include='object').columns
string_columns_empty_check = df[string_columns].apply(lambda x: (x.str.strip() == '').sum())
string_columns_empty_check
# => TotalCharges has 11 empty strings which needs to be handled.

### Duplicate analysis

In [ ]:
# deduplication
print("Duplicate rows:", df.duplicated().sum())

# check for multiple records with same customerID
print("Duplicate customer IDs:", df.duplicated(subset='customerID', keep=False).sum())

cid_unique_count = df['customerID'].nunique()
print("Unique customer IDs:", cid_unique_count)
print("Total rows:", df.shape[0])
# => All customerID are unique, no duplicates found.

### Numerical vs Categorial identification

In [ ]:
# finalize numerical and categorical columns
numerical_columns = [
    'tenure', 'MonthlyCharges', 'TotalCharges'
]
categorical_columns = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod'
]
target = 'Churn'

### Target-variable analysis

In [ ]:
# target analysis
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True).mul(100).round(2))
# => 73.46% of customers are not churned and 26.54% of customers are churned, data is imbalanced (can be handled in training).

### Data cleaning & preprocessing

In [ ]:
df['TotalCharges'].isnull().sum()

In [ ]:
# Fix TotalCharges: convert to numeric, coercing blank strings to NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].isnull().sum()
# => 11 null values found in TotalCharges after conversion, which were originally empty strings.

In [ ]:
# These are almost always customers with tenure == 0 (brand new, no charges yet) — verify:
print(df.loc[df['TotalCharges'].isnull(), 'tenure'].value_counts())
# Impute with 0 since it reflects reality (no charges accrued yet)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

In [ ]:
# customerID is a unique identifier — no predictive value, drop it
df.drop(columns=['customerID'], inplace=True)

In [ ]:
# Defensive: strip stray whitespace from all string columns
obj_cols = df.select_dtypes(include='str').columns
df[obj_cols] = df[obj_cols].apply(lambda col: col.str.strip())

In [ ]:
df.info()

### Encoding categorical variables

In [ ]:
# identify columns
print("Unique values")
for col in df.columns:
    print(f"- {col}")
    unique_values = df[col].nunique()
    print(f"  - Count: {unique_values}")
    if unique_values <= 10:
        print(f"  - Values: {df[col].unique().tolist()}")


In [ ]:
YN_MAP = {'Yes': 1, 'No': 0}
GENDER_MAP = {'Male': 1, 'Female': 0}

In [ ]:
# Binary Yes/No columns -> 0/1
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
# SeniorCitizen is already 0/1, so no need to convert it.
for col in binary_cols:
    df[col] = df[col].map(YN_MAP)

df['gender'] = df['gender'].map(GENDER_MAP)
df['Churn'] = df['Churn'].map(YN_MAP)

In [ ]:
# multi category columns
multi_cat_cols = [
    'MultipleLines', 'InternetService', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'Contract',
    'PaymentMethod'
]
df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=False)

In [ ]:
df.info()

In [ ]:
# cast all boolean columns to int (0/1) for consistency
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)
df.info()

## 2. Exploratory data analysis (EDA)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

%matplotlib inline
sns.set_style('whitegrid')

# Rebuild a clean, unencoded copy for EDA (reload raw + reapply only the cleaning steps)
df_eda = pd.read_csv('../data/TelcoCustomerChurn.csv')  # adjust to your actual path
df_eda['TotalCharges'] = pd.to_numeric(df_eda['TotalCharges'], errors='coerce').fillna(0)
obj_cols = df_eda.select_dtypes(include='str').columns
df_eda[obj_cols] = df_eda[obj_cols].apply(lambda col: col.str.strip())

In [ ]:
df_eda

### Churn distribution

In [ ]:
# churn distribution
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df_eda, x='Churn', hue='Churn', palette=['#4C72B0', '#DD8452'], legend=False)
total = len(df_eda)
for p in ax.patches:
    pct = f'{100 * p.get_height() / total:.1f}%'
    ax.annotate(pct, (p.get_x() + p.get_width() / 2, p.get_height()), ha='center', va='bottom')
plt.title('Customer Churn Distribution')
plt.show()

Insight: This dataset typically runs around a 26–27% churn rate. If your ~4k-row subset lands near that, you have a meaningfully imbalanced target — confirms the stratify=y split from step 1 was the right call, and accuracy alone won't be a fair model metric later (watch precision/recall/F1 or ROC-AUC instead).

### Customer/service characteristics

In [ ]:
cat_cols = ['Contract', 'InternetService', 'PaymentMethod', 'PaperlessBilling', 'Partner', 'Dependents']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.flatten(), cat_cols):
    sns.countplot(data=df_eda, x=col, hue=col, ax=ax, order=df_eda[col].value_counts().index, legend=False)
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

Insight: Shows what your customer base actually looks like. Typically most customers sit on month-to-month contracts and roughly a third use fiber optic internet — worth noting because (see next chart) those same segments tend to be the highest-churn ones, meaning a large share of the base already sits in the "at-risk" profile.

### Churn rate by contract type

In [ ]:
plt.figure(figsize=(6, 4))
churn_by_contract = (df_eda.groupby('Contract')['Churn']
                     .apply(lambda x: (x == 'Yes').mean() * 100)
                     .sort_values(ascending=False))
sns.barplot(x=churn_by_contract.index, y=churn_by_contract.values, hue=churn_by_contract.index, legend=False)
plt.ylabel('Churn Rate (%)')
plt.title('Churn Rate by Contract Type')
plt.show()

Insight: Month-to-month customers usually churn at several times the rate of one/two-year contract holders. Contract length is generally the single strongest categorical churn driver — commitment period matters more than almost any service feature.

### Churn rate by internet service & payment method

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

churn_by_internet = df_eda.groupby('InternetService')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
sns.barplot(x=churn_by_internet.index, y=churn_by_internet.values, hue=churn_by_internet.index, ax=axes[0], legend=False)
axes[0].set_title('Churn Rate by Internet Service')
axes[0].set_ylabel('Churn Rate (%)')

churn_by_payment = (df_eda.groupby('PaymentMethod')['Churn']
                    .apply(lambda x: (x == 'Yes').mean() * 100)
                    .sort_values(ascending=False))
sns.barplot(x=churn_by_payment.index, y=churn_by_payment.values, hue=churn_by_payment.index, ax=axes[1], legend=False)
axes[1].set_title('Churn Rate by Payment Method')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

Insight: Fiber optic customers typically churn more than DSL customers (often pricing or reliability complaints), and electronic-check payers usually churn noticeably more than customers on automatic bank/card payments. That payment-method gap is a well-known signal in this dataset — manual payment tends to correlate with lower commitment or more billing friction.

### Numerical variable distributions by churn

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, num_cols):
    sns.histplot(data=df_eda, x=col, hue='Churn', kde=True, element='step', ax=ax)
    ax.set_title(f'{col} distribution by churn')
plt.tight_layout()
plt.show()

Insight: Churners typically cluster heavily at low tenure — the first several months are usually the highest-risk window for losing a customer. Low TotalCharges among churners is mostly a side effect of that short tenure rather than an independent driver, since it accumulates over time.

### Correlation heatmap (numerical features + churn)

In [ ]:
plt.figure(figsize=(6, 5))
corr_df = df_eda[['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']].copy()
corr_df['Churn'] = df_eda['Churn'].map({'Yes': 1, 'No': 0})
sns.heatmap(corr_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

Insight: tenure usually shows the strongest negative correlation with churn among the numeric features. TotalCharges tends to correlate very highly with tenure itself (it's cumulative by construction), so the two carry overlapping information — flag this now, since it matters for multicollinearity if you use a linear model later.

## 3. Feature Engineering

### Tenure Group

In [ ]:
# Bucketed the continuous tenure column into four ordinal stages that roughly map to a customer's relationship lifecycle — brand new, settling in, established, long-term.
# Why it's useful: Your EDA already showed churn is heavily concentrated in low tenure and drops off sharply after — that's not a straight-line relationship, it's closer to a cliff in year one. Tree models can pick that up from raw tenure fine, but a linear or logistic model can't represent a "cliff" from a single continuous coefficient. Binning makes that non-linearity explicit for any model type, and it's also a much easier feature to act on in business terms ("customers under 1 year are the priority retention segment") than a raw month count.

def tenure_group(months: int):
    if months <= 12:
        return '0-1yr'
    elif months <= 24:
        return '1-2yr'
    elif months <= 48:
        return '2-4yr'
    else:
        return '4yr+'

df['TenureGroup'] = df['tenure'].apply(tenure_group)

### Total Services Subscribed

In [ ]:
service_cols = [
    'PhoneService',
    'MultipleLines_Yes',
    'InternetService_DSL',
    'InternetService_Fiber optic',
    'OnlineSecurity_Yes',
    'OnlineBackup_Yes',
    'DeviceProtection_Yes',
    'TechSupport_Yes',
    'StreamingTV_Yes',
    'StreamingMovies_Yes'
]
def count_services(row):
    return sum(
        row[col] for col in service_cols
    )

df['TotalServices'] = df.apply(count_services, axis=1)
df

### Charge Deviation (recent bill change)

In [ ]:
df['AvgMonthlyCharge'] = df['TotalCharges'] / df['tenure'].replace(0, 1) # handle division by zero for new customers with tenure 0
df['ChargeDeviation'] = df['MonthlyCharges'] - df['AvgMonthlyCharge']

- `AvgMonthlyCharge` is each customer's historical average spend (`TotalCharges` / `tenure`, with `tenure=0` treated as 1 to avoid division by zero — those customers correctly get an average of 0, matching their $0 total charges).  
- `ChargeDeviation` is the gap between what they're currently being billed (`MonthlyCharges`) and that historical average.  
- A positive value means the customer's current bill is higher than what they've historically paid — a promo expiring, a plan change, or a price increase. `MonthlyCharges` or `TotalCharges` alone only show the level of spend, not whether it just changed, and a sudden increase is a classic churn trigger even for customers whose absolute bill isn't unusually high. This feature is designed to flag exactly that group.

In [ ]:
df

### Encode new features

In [ ]:
df = pd.get_dummies(df, columns=['TenureGroup'], drop_first=False)
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)
df

## 4. Model Development

### Train Test Split

In [ ]:
df.info()

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

# Check the distribution of the target variable in the subsets
print("\nTraining Set:")
print("X_train", X_train.shape)
print("y_train", y_train.value_counts())
print("y_train %", y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTest Set:")
print("X_test", X_test.shape)
print("y_test", y_test.value_counts())
print("y_test %", y_test.value_counts(normalize=True).mul(100).round(2))

# => Class imbalance is preserved in both training and test sets
# => Good for model evaluation

### Build

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    ConfusionMatrixDisplay
)
from sklearn.tree import DecisionTreeClassifier

### Configuration 1: Baseline (unconstrained) tree

In [ ]:
dt_baseline = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt_baseline.fit(X_train, y_train)

train_acc1 = dt_baseline.score(X_train, y_train)
test_acc1 = dt_baseline.score(X_test, y_test)
print(f"Baseline — Train acc: {train_acc1:.3f} | Test acc: {test_acc1:.3f}")

y_pred1 = dt_baseline.predict(X_test)
y_proba1 = dt_baseline.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_pred1, target_names=['No Churn', 'Churn']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba1):.3f}")

Letting the tree grow with no constraints (max_depth=None) almost always drives train accuracy very close to 1.0 while test accuracy sits meaningfully lower — that gap is the tree memorizing noise in the training data rather than learning generalizable churn patterns.

### Configuration 2: Constrained tree, class-weight balanced

In [ ]:
dt_constrained = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=30,
    class_weight='balanced',
    random_state=RANDOM_STATE
)
dt_constrained.fit(X_train, y_train)

train_acc2 = dt_constrained.score(X_train, y_train)
test_acc2 = dt_constrained.score(X_test, y_test)
print(f"Constrained — Train acc: {train_acc2:.3f} | Test acc: {test_acc2:.3f}")

y_pred2 = dt_constrained.predict(X_test)
y_proba2 = dt_constrained.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_pred2, target_names=['No Churn', 'Churn']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba2):.3f}")

`max_depth=5` and `min_samples_leaf=30` cap how much the tree can twist itself around individual training rows, which should shrink the train/test gap from config 1. `class_weight='balanced'` directly addresses the ~27% churn class imbalance you found earlier — without it, a tree can rack up decent overall accuracy just by leaning toward predicting "No Churn" which is exactly the wrong bias for a retention use case where missing an actual churner is the costly error.

### Configuration 3: grid-searched tree

Two hand-picked configs satisfy the requirement, but a small grid search gives you a defensible "best of many" candidate rather than two arbitrary guesses:

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_leaf': [10, 20, 30, 50],
    'criterion': ['gini', 'entropy'],
    'class_weight': ['balanced', None]
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid,
    cv=5,
    scoring='roc_auc',   # picked over accuracy because the target is imbalanced
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV ROC-AUC:", grid_search.best_score_)

dt_tuned = grid_search.best_estimator_
y_pred3 = dt_tuned.predict(X_test)
y_proba3 = dt_tuned.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_pred3, target_names=['No Churn', 'Churn']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba3):.3f}")

# confusion matrix
cm = confusion_matrix(y_test, y_pred3)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix — Nos')
plt.show()
cm_pct = confusion_matrix(y_test, y_pred3, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm_pct, display_labels=['No Churn', 'Churn'])
disp.plot(cmap='Blues', values_format='.2%')
plt.title('Confusion Matrix — %')
plt.show()

`scoring='roc_auc'` rather than `'accuracy'` because accuracy is a misleading optimization target on an imbalanced class — you can swap in 'f1' or 'recall' instead if the business priority is "catch as many likely churners as possible, even at the cost of some false alarms," which is a legitimate framing for a retention team.

How to read the four cells for this problem
- Top-left (True Negative): correctly predicted "won't churn."
- Top-right (False Positive): predicted churn, but they stayed. Cost = a retention offer spent on someone who didn't need it — annoying, but cheap.
- Bottom-left (False Negative): predicted no churn, but they left. This is the expensive mistake — a customer walks out the door with no intervention because the model missed them.
- Bottom-right (True Positive): correctly caught a churner in time to act.

**Accuracy**: "Out of everyone, what fraction did I classify correctly?" This is the metric most likely to mislead you here — with ~27% churn, a model that just predicts "No Churn" for every single customer would still score around 73% accuracy while catching zero actual churners. It treats every cell in the matrix as equally important, which isn't true for this problem.
`Accuracy=(TP+TN)/(TP+TN+FP+FN)`​

**Precision**: "Of the customers I flagged as likely to churn, how many actually churned?" Low precision means you're spending retention budget (calls, discounts, offers) on customers who were never going to leave — the false-positive cost from your confusion matrix discussion.
`Precision=TP/(TP+FP)​`

**Recall**: "Of the customers who actually churned, how many did I catch?" This is the one to weight most heavily for your use case — it directly measures the false-negative cell, which you already identified as the expensive mistake: a churner walks out the door with zero intervention because the model missed them.
`Recall=TP/(TP+FN)​`

**F1-score**: The harmonic mean of precision and recall — it only stays high if both are reasonably high, so it punishes a model that inflates recall by flagging almost everyone as a churn risk (which would tank precision). Useful as a single number when you need to rank models but don't want to look at precision and recall separately every time.
`F1= 2 × (Precision * Recall) / (Precision + Recall​)`



### Compare models

In [ ]:
def summarize(name, model):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    report = classification_report(y_test, y_pred, output_dict=True)
    return {
        'Model': name,
        'Train Acc': model.score(X_train, y_train),
        'Test Acc': accuracy_score(y_test, y_pred),
        'Precision (Churn)': report['1']['precision'],
        'Recall (Churn)': report['1']['recall'],
        'F1 (Churn)': report['1']['f1-score'],
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    }

comparison = pd.DataFrame([
    summarize('Baseline (unconstrained)', dt_baseline),
    summarize('Constrained (manual)', dt_constrained),
    summarize('Grid-searched (tuned)', dt_tuned),
])
comparison

Selecting and justifying the final model

Don't just pick whichever row has the highest test accuracy — for this problem, judge the three candidates against:

Generalization gap (Train Acc − Test Acc) — the baseline will almost certainly have the widest gap; a smaller gap means the model learned real patterns instead of memorizing training rows.

Recall/F1 on the Churn class, not overall accuracy — a model that predicts "No Churn" for nearly everyone can still post a deceptively high accuracy on an imbalanced target. Since a missed churner is usually more costly to the business than a false alarm, weight recall and F1 for the Churn class heavily.

ROC-AUC — a threshold-independent view of how well the model ranks churners above non-churners, useful since you can adjust the decision threshold later depending on retention-campaign capacity.

Interpretability — one advantage of decision trees over black-box models is that stakeholders can read the rules. If the grid search picks max_depth=None or something very deep, you trade away that interpretability for a small accuracy gain — worth explicitly deciding whether that trade is worth it.

Run the comparison table above and pick using this rule of thumb: discard the baseline outright if its train/test gap is large (it's overfitting almost by definition), then between the constrained and tuned models, pick whichever has the better Recall/F1/ROC-AUC on the Churn class — unless the gap is small and the constrained tree's shallower depth is worth more to you for interpretability. Write the actual numbers from your run into that justification when you report it; I can help interpret the specific output once you have it.

### Visualize Model

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.tree import export_text, plot_tree

# final_model = dt_baseline  # swap in whichever model you actually selected from the comparison
# final_model = dt_constrained  # swap in whichever model you actually selected from the comparison
final_model = dt_tuned  # swap in whichever model you actually selected from the comparison

plt.figure(figsize=(20, 10))
plot_tree(
    final_model,
    feature_names=X_train.columns,
    class_names=['No Churn', 'Churn'],
    filled=True,
    rounded=True,
    fontsize=8,
    # max_depth=3   # caps what's *drawn*, not the model itself — full tree is usually unreadable past a few levels
)
plt.title('Final Decision Tree')
plt.show()

> **Next: Bagging, Boosting, Randomforest, Logistic Regression, Optuna hyperparameter tuning**

## 5. Model Interpretation
Explain what drives the model's churn predictions. 

### Feature importance

In [ ]:
# You already have feature_importances_ from the final model, but worth being precise about what it actually measures: for a decision tree, it's the total reduction in impurity (Gini/entropy) each feature contributes across all its splits, normalized to sum to 1. It's a "how much did this feature help the tree separate classes" score, not a causal effect.
importances = pd.Series(final_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
importances.head(10).plot(kind='barh')
plt.gca().invert_yaxis()
plt.xlabel('Gini Importance')
plt.title('Top 10 Feature Importances')
plt.tight_layout()
plt.show()

print(importances.head(10))
print(f"\nTop 10 features account for {importances.head(10).sum()*100:.1f}% of total importance")

In [ ]:
# One caveat worth flagging in your write-up: impurity-based importance can be biased toward continuous or high-cardinality features (like MonthlyCharges or TotalCharges, which have far more possible split points than a binary one-hot column). A cross-check with permutation importance — which measures the actual drop in test-set performance when a feature is randomly shuffled — is a more trustworthy second opinion:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    final_model, X_test, y_test, n_repeats=10, random_state=42, scoring='roc_auc'
)
perm_importances = pd.Series(perm_result.importances_mean, index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
perm_importances.head(10).plot(kind='barh')
plt.gca().invert_yaxis()
plt.xlabel('Mean drop in ROC-AUC when shuffled')
plt.title('Top 10 Permutation Importances (Test Set)')
plt.tight_layout()
plt.show()

print(perm_importances.head(10))

### Top features influencing churn

In [ ]:
# Put both rankings side by side — if they roughly agree, that's good evidence the model is relying on genuine signal rather than an artifact of how impurity importance is calculated:
comparison_df = pd.DataFrame({
    'Gini Importance': importances,
    'Permutation Importance': perm_importances
}).sort_values('Permutation Importance', ascending=False)

print(comparison_df.head(10))

### Decision tree visualization & interpretation

In [ ]:
# You already have the plotting code from earlier in this thread (plot_tree / export_text) — for interpretation specifically, the most useful thing to pull out is what the root split and first couple of levels actually say, since that's where the tree is making its biggest, most generalizable distinctions:
from sklearn.tree import export_text

rules = export_text(final_model, feature_names=list(X_train.columns), max_depth=3)
print(rules)
# Read this top-down as a plain-language decision path — e.g. "if Contract_Month-to-month == 1 and tenure <= X, predicted class is Churn." The higher up a feature appears, the more customers it's splitting on and generally the more influential it is, which should roughly (not always exactly) track the importance rankings above.

### Key findings — write-up template

TODO:

The model's top predictors are [list your actual top 3–4 from the comparison table]. This is [consistent / inconsistent] with the EDA and feature engineering:

If Contract ranks highly: confirms the EDA finding that month-to-month customers churn at a much higher rate than annual contract holders — likely the single strongest lever available to a retention team (incentivizing longer commitments).
If tenure / TenureGroup ranks highly: confirms the "early tenure cliff" — new customers are the highest-risk population, supporting the binned feature's rationale.
If TotalServices ranks highly: supports the "bundling = stickiness" hypothesis from feature engineering — customers with fewer add-on services are easier to lose.
If ChargeDeviation ranks highly: supports the idea that a recent bill increase, not just the absolute charge level, is a churn trigger.

Overall, [the model's decision logic mirrors what the EDA already suggested / the model surfaced a pattern the EDA didn't flag, specifically ___]. [If the former:] that's a good sign — it means the tree is learning genuine, explainable business drivers rather than overfitting to noise in a ~4k-row dataset.